# ChurnIQ — Exploratory Data Analysis & Machine Learning Pipeline
### Customer Retention Intelligence Platform
**Author:** B.Tech AI & Data Science Candidate  
**Dataset:** Telco Customer Churn (7,043 instances, 21 attributes)

---

## 1. Problem Formulation & Business Objectives
Acquiring a new customer can cost 5x to 7x more than retaining an existing customer in subscription businesses.
The objective of **ChurnIQ** is to predict which customers are at risk of churning, identify the key drivers behind customer churn, and inform proactive customer retention strategies.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix

# Set plotting aesthetic
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'Inter'
print('Libraries imported successfully.')

## 2. Data Ingestion & Sanitization
Inspect data types, handle missing values, and transform TotalCharges.

In [2]:
df = pd.read_csv('../data/telco_customer_churn.csv')
print(f"Dataset shape: {df.shape}")

# TotalCharges has 11 whitespace strings for tenure=0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce').fillna(0.0)
df['SeniorCitizen'] = df['SeniorCitizen'].astype(int)
df['ChurnBinary'] = df['Churn'].map({'Yes': 1, 'No': 0})

churn_rate = df['ChurnBinary'].mean()
print(f"Baseline Churn Rate: {churn_rate * 100:.2f}%")
df.head(3)

## 3. Exploratory Data Analysis & Churn Drivers
Analyzing relationship between contract length, tenure, internet service, and churn rate.

In [3]:
# Contract vs Churn
contract_churn = df.groupby('Contract')['ChurnBinary'].agg(['count', 'mean']).reset_index()
contract_churn.columns = ['Contract', 'Total Customers', 'Churn Rate']
contract_churn['Churn Rate %'] = contract_churn['Churn Rate'] * 100
print("Churn by Contract Type:")
print(contract_churn[['Contract', 'Total Customers', 'Churn Rate %']])

# Internet Service vs Churn
internet_churn = df.groupby('InternetService')['ChurnBinary'].agg(['count', 'mean']).reset_index()
internet_churn['Churn Rate %'] = internet_churn['mean'] * 100
print("\nChurn by Internet Service:")
print(internet_churn[['InternetService', 'count', 'Churn Rate %']])

## 4. Modeling Pipeline & Evaluation (No Data Leakage)
Using stratified splitting and ColumnTransformer inside Scikit-learn Pipeline.

In [4]:
from ml.preprocessing import ALL_FEATURE_COLUMNS, build_preprocessor

X = df[ALL_FEATURE_COLUMNS]
y = df['ChurnBinary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = build_preprocessor()
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print(f"Holdout ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred, target_names=['Retained', 'Churned']))